In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('jobs_in_data.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))
print("\nMissing values:")
print(df.isnull().sum())

Shape: (9355, 12)

Columns: ['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']

First 3 rows:
   work_year             job_title                    job_category  \
0       2023  Data DevOps Engineer                Data Engineering   
1       2023        Data Architect  Data Architecture and Modeling   
2       2023        Data Architect  Data Architecture and Modeling   

  salary_currency  salary  salary_in_usd employee_residence experience_level  \
0             EUR   88000          95012            Germany        Mid-level   
1             USD  186000         186000      United States           Senior   
2             USD   81800          81800      United States           Senior   

  employment_type work_setting company_location company_size  
0       Full-time       Hybrid          Germany            L  
1       Full-time    In-per

In [5]:
import sqlite3

conn = sqlite3.connect('jobs_data.db')
df.to_sql('jobs', conn, if_exists='replace', index=False)

print("=== TOP JOB CATEGORIES ===")
q1 = pd.read_sql_query("""
    SELECT job_category,
           COUNT(*) as total_jobs,
           ROUND(AVG(salary_in_usd), 0) as avg_salary
    FROM jobs
    GROUP BY job_category
    ORDER BY total_jobs DESC
""", conn)
print(q1)

print("\n=== SALARY BY EXPERIENCE LEVEL ===")
q2 = pd.read_sql_query("""
    SELECT experience_level,
           COUNT(*) as total_jobs,
           ROUND(AVG(salary_in_usd), 0) as avg_salary,
           ROUND(MAX(salary_in_usd), 0) as max_salary
    FROM jobs
    GROUP BY experience_level
    ORDER BY avg_salary DESC
""", conn)
print(q2)

print("\n=== TOP 5 COMPANY LOCATIONS ===")
q3 = pd.read_sql_query("""
    SELECT company_location,
           COUNT(*) as total_jobs,
           ROUND(AVG(salary_in_usd), 0) as avg_salary
    FROM jobs
    GROUP BY company_location
    ORDER BY total_jobs DESC
    LIMIT 5
""", conn)
print(q3)

conn.close()
print("\nSQL Analysis Complete!")

=== TOP JOB CATEGORIES ===
                     job_category  total_jobs  avg_salary
0       Data Science and Research        3014    163759.0
1                Data Engineering        2260    146198.0
2                   Data Analysis        1457    108506.0
3         Machine Learning and AI        1428    178926.0
4       Leadership and Management         503    145476.0
5            BI and Visualization         313    135092.0
6  Data Architecture and Modeling         259    156002.0
7    Data Management and Strategy          61    103140.0
8     Data Quality and Operations          55    100879.0
9              Cloud and Database           5    155000.0

=== SALARY BY EXPERIENCE LEVEL ===
  experience_level  total_jobs  avg_salary  max_salary
0        Executive         281    189463.0    416000.0
1           Senior        6709    162356.0    412000.0
2        Mid-level        1869    117524.0    450000.0
3      Entry-level         496     88535.0    281700.0

=== TOP 5 COMPANY LOCAT

In [6]:
df_clean = df[['job_title', 'job_category', 'salary_in_usd',
               'experience_level', 'employment_type',
               'work_setting', 'company_location',
               'company_size', 'work_year']].copy()

df_clean['experience_level'] = df_clean['experience_level'].replace({
    'EN': 'Entry-level',
    'MI': 'Mid-level',
    'SE': 'Senior',
    'EX': 'Executive'
})

df_clean['company_size'] = df_clean['company_size'].replace({
    'S': 'Small',
    'M': 'Medium',
    'L': 'Large'
})

print("Cleaned shape:", df_clean.shape)
print("\nExperience levels:", df_clean['experience_level'].unique())
print("Company sizes:", df_clean['company_size'].unique())
print("\nCleaning Complete!")

Cleaned shape: (9355, 9)

Experience levels: ['Mid-level' 'Senior' 'Executive' 'Entry-level']
Company sizes: ['Large' 'Medium' 'Small']

Cleaning Complete!


In [7]:
import plotly.graph_objects as go
import networkx as nx
import plotly.io as pio
pio.renderers.default = "colab"

skills = ['Python', 'SQL', 'Tableau', 'Power BI', 'Machine Learning',
          'Deep Learning', 'NLP', 'Excel', 'R', 'Spark']

edges = [
    ('Python', 'Machine Learning'), ('Python', 'Deep Learning'),
    ('Python', 'NLP'), ('Python', 'SQL'), ('Python', 'Spark'),
    ('SQL', 'Tableau'), ('SQL', 'Power BI'), ('SQL', 'Excel'),
    ('Machine Learning', 'Deep Learning'), ('Machine Learning', 'NLP'),
    ('Tableau', 'Power BI'), ('R', 'Machine Learning'),
    ('Spark', 'SQL'), ('Excel', 'Power BI')
]

G = nx.Graph()
G.add_nodes_from(skills)
G.add_edges_from(edges)
pos = nx.spring_layout(G, seed=42)

edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=2, color='#4361ee'),
    hoverinfo='none',
    mode='lines'
)

node_x = [pos[node][0] for node in G.nodes()]
node_y = [pos[node][1] for node in G.nodes()]
node_degrees = [G.degree(node) * 15 for node in G.nodes()]

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=list(G.nodes()),
    textposition='top center',
    textfont=dict(color='white', size=12),
    marker=dict(
        size=node_degrees,
        color=['#f72585', '#b5179e', '#7209b7', '#4361ee',
               '#4cc9f0', '#f72585', '#b5179e', '#7209b7',
               '#4361ee', '#4cc9f0'],
        line=dict(width=2, color='white')
    )
)

fig1 = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title='🔗 Data Science Skills Network',
        titlefont=dict(size=22, color='white'),
        title_x=0.5,
        paper_bgcolor='#0d0d0d',
        plot_bgcolor='#1a1a2e',
        height=600,
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
)

fig1.show()

In [8]:
import plotly.express as px

yearly_jobs = df_clean.groupby(
    ['work_year', 'job_category']
).size().reset_index(name='job_count')

fig2 = px.bar(
    yearly_jobs,
    x='job_count',
    y='job_category',
    color='job_category',
    animation_frame='work_year',
    orientation='h',
    title='🏁 Job Category Growth Race 2020-2023',
    color_discrete_sequence=[
        '#f72585', '#b5179e', '#7209b7',
        '#4361ee', '#4cc9f0', '#3a0ca3',
        '#560bad', '#4895ef', '#3f37c9', '#480ca8'
    ],
    range_x=[0, 3200]
)

fig2.update_layout(
    title_font_size=22,
    title_x=0.5,
    paper_bgcolor='#0d0d0d',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', size=12),
    height=600,
    showlegend=False,
    yaxis={'categoryorder': 'total ascending'}
)
fig2.update_xaxes(gridcolor='#333366', color='white')
fig2.update_yaxes(gridcolor='#333366', color='white')

fig2.show()

In [9]:
import plotly.express as px

df_parallel = df_clean.copy()

exp_map = {'Entry-level': 1, 'Mid-level': 2, 'Senior': 3, 'Executive': 4}
size_map = {'Small': 1, 'Medium': 2, 'Large': 3}
setting_map = {'In-person': 1, 'Hybrid': 2, 'Remote': 3}

df_parallel['exp_numeric'] = df_parallel['experience_level'].map(exp_map)
df_parallel['size_numeric'] = df_parallel['company_size'].map(size_map)
df_parallel['setting_numeric'] = df_parallel['work_setting'].map(setting_map)

fig3 = px.parallel_coordinates(
    df_parallel,
    dimensions=['exp_numeric', 'salary_in_usd',
                'size_numeric', 'setting_numeric', 'work_year'],
    color='salary_in_usd',
    color_continuous_scale=[
        '#0d0d0d', '#7209b7', '#b5179e',
        '#f72585', '#4cc9f0'
    ],
    labels={
        'exp_numeric': 'Experience',
        'salary_in_usd': 'Salary (USD)',
        'size_numeric': 'Company Size',
        'setting_numeric': 'Work Setting',
        'work_year': 'Year'
    },
    title='⚡ Parallel Coordinates — Job Market Patterns'
)

fig3.update_layout(
    title_font_size=22,
    title_x=0.5,
    paper_bgcolor='#0d0d0d',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', size=12),
    height=600,
    coloraxis_colorbar=dict(
        title='Salary',
        tickfont=dict(color='white')
    )
)

fig3.show()

In [10]:
import plotly.express as px

heatmap_data = df_clean.groupby(
    ['job_category', 'experience_level']
)['salary_in_usd'].mean().reset_index()

heatmap_pivot = heatmap_data.pivot(
    index='job_category',
    columns='experience_level',
    values='salary_in_usd'
)

fig4 = px.imshow(
    heatmap_pivot,
    title='🔥 Avg Salary — Job Category vs Experience Level',
    color_continuous_scale=[
        '#0d0d0d', '#3a0ca3', '#7209b7',
        '#b5179e', '#f72585', '#4cc9f0'
    ],
    aspect='auto',
    text_auto='.0f'
)

fig4.update_layout(
    title_font_size=22,
    title_x=0.5,
    paper_bgcolor='#0d0d0d',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', size=11),
    height=550
)

fig4.update_xaxes(color='white')
fig4.update_yaxes(color='white')
fig4.show()

In [11]:
import plotly.express as px

bubble_data = df_clean.groupby('job_category').agg(
    avg_salary=('salary_in_usd', 'mean'),
    total_jobs=('job_title', 'count'),
    avg_year=('work_year', 'mean')
).reset_index()

fig5 = px.scatter(
    bubble_data,
    x='avg_salary',
    y='job_category',
    size='total_jobs',
    color='avg_salary',
    title='💫 Job Category — Salary vs Demand',
    color_continuous_scale=[
        '#3a0ca3', '#7209b7', '#b5179e',
        '#f72585', '#4cc9f0'
    ],
    size_max=80,
    hover_data=['total_jobs'],
    text='job_category'
)

fig5.update_traces(
    textposition='middle right',
    textfont=dict(color='white', size=11)
)

fig5.update_layout(
    title_font_size=22,
    title_x=0.5,
    paper_bgcolor='#0d0d0d',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white', size=12),
    height=600,
    showlegend=False,
    coloraxis_showscale=False
)
fig5.update_xaxes(gridcolor='#333366', color='white')
fig5.update_yaxes(gridcolor='#333366', color='white', showticklabels=False)
fig5.show()

In [12]:
df_clean.to_csv('job_market_clean.csv', index=False)
print("CSV exported!")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

CSV exported!
Rows: 9355
Columns: 9
